# DS-3002 Data Mining - Assignment 2
## From Clicks to Loyalty: Customer Intelligence Pipeline
This notebook solves the full assignment using UCI Online Retail II data. It covers preprocessing, FP-Growth, class association rules (CARs), and sequential mining for churn-warning trigger design.

In [45]:
import pandas as pd

df1 = pd.read_excel("online_retail_II.xlsx", sheet_name=0)
df2 = pd.read_excel("online_retail_II.xlsx", sheet_name=1)

df = pd.concat([df1, df2])
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


---
## Required Preprocessing 

In [46]:
# Remove cancellations
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# Remove invalid rows
df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]

# Remove null customers
df = df.dropna(subset=['Customer ID'])

# Keep UK only
df = df[df['Country'] == 'United Kingdom']

df.reset_index(inplace=True)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 725250 entries, 0 to 725249
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   index        725250 non-null  int64         
 1   Invoice      725250 non-null  object        
 2   StockCode    725250 non-null  object        
 3   Description  725250 non-null  object        
 4   Quantity     725250 non-null  int64         
 5   InvoiceDate  725250 non-null  datetime64[us]
 6   Price        725250 non-null  float64       
 7   Customer ID  725250 non-null  float64       
 8   Country      725250 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(2), object(3), str(1)
memory usage: 59.6+ MB


In [72]:
print("Invoices:", df['Invoice'].nunique())
print("Customers:", df['Customer ID'].nunique())
print("Products:", df['StockCode'].nunique())
print("Date Range:", df['InvoiceDate'].min(), df['InvoiceDate'].max())

Invoices: 33541
Customers: 5350
Products: 4616
Date Range: 2009-12-01 07:45:00 2011-12-09 12:49:00


### Preprocessing Outcome
After cleaning and UK filtering, the dataset is ready for both invoice-level mining (Parts A/B) and customer journey sequence mining (Part C). The metrics printed above are used as the baseline in the report.

In [73]:
basket = df.groupby('Invoice')['StockCode'].nunique()
print(basket.head())
print("Avg Basket Size:", basket.mean())

Invoice
489434     8
489435     4
489436    19
489437    23
489438    17
Name: StockCode, dtype: int64
Avg Basket Size: 20.571032467726067


---
## Part A: FP-Growth - Efficient Co-Purchase Mining 
This section demonstrates FP-Tree intuition on a small sample and then scales to full FP-Growth with minsup = 2% and minconf = 50%.

In [71]:
#A1
#=== 6 Sample Invoices for FP-Tree Construction ===
#T1 (Invoice 489434): 21523, 22041, 22064, 79323P, 79323W
#T2 (Invoice 489435): 22195, 22349, 22350, 22353
#T3 (Invoice 489436): 22119, 22142, 22295, 35004B, 82582
#T4 (Invoice 489437): 20695, 21364, 21912, 35400, 84970S
#T5 (Invoice 489438): 21329, 21411, 84031A, 84519B, 85183A
#T6 (Invoice 489441): 22111, 22138, 22321, 84029E
print('''
      Root
        ├── 21523 -> 22041 -> 22064 -> 79323P -> 79323W
        ├── 22195 -> 22349 -> 22350 -> 22353
        ├── 22119 -> 22142 -> 22295 -> 35004B -> 82582
        ├── 20695 -> 21364 -> 21912 -> 35400 -> 84970S
        ├── 21329 -> 21411 -> 84031A-> 84519B -> 85183A
        └── 22111 -> 22138 -> 22321 -> 84029E
      ''')


      Root
        ├── 21523 → 22041 → 22064 → 79323P → 79323W
        ├── 22195 → 22349 → 22350 → 22353
        ├── 22119 → 22142 → 22295 → 35004B → 82582
        ├── 20695 → 21364 → 21912 → 35400 → 84970S
        ├── 21329 → 21411 → 84031A → 84519B → 85183A
        └── 22111 → 22138 → 22321 → 84029E
      


In [70]:
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

df['StockCode'] = df['StockCode'].apply(lambda x: str(x))
# Build transaction matrix
basket = df.groupby('Invoice')['StockCode'].apply(list)
display(basket.head(6))
te = TransactionEncoder()
te_array = te.fit_transform(basket)
df_te = pd.DataFrame(te_array, columns=te.columns_)

# Run FP-Growth
frequent_itemsets = fpgrowth(df_te, min_support=0.02, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)


Invoice
489434    [85048, 79323P, 79323W, 22041, 21232, 22064, 2...
489435                         [22350, 22349, 22195, 22353]
489436    [48173C, 21755, 21754, 84879, 22119, 22142, 22...
489437    [22143, 22145, 22130, 21364, 21360, 21351, 213...
489438    [21329, 21252, 21100, 21033, 20711, 21410, 214...
489440                                       [22350, 22349]
Name: StockCode, dtype: object

In [74]:
# Get counts per size
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets.groupby('length').size()

length
1    193
2     28
dtype: int64

In [52]:
# Create a StockCode -> Description mapping from your dataframe
stock_desc = df.drop_duplicates('StockCode').set_index('StockCode')['Description'].to_dict()

# Function to convert frozenset of StockCodes to readable names
def decode_itemset(itemset):
    return {stock_desc.get(item, item) for item in itemset}

# Apply to your rules dataframe
rules['antecedents_readable'] = rules['antecedents'].apply(decode_itemset)
rules['consequents_readable'] = rules['consequents'].apply(decode_itemset)

# Display top 15 rules with readable names
top15 = rules.sort_values('lift', ascending=False).head(16).reset_index(drop=True)
top15[['antecedents_readable', 'consequents_readable', 'support', 'confidence', 'lift']]

,antecedents_readable,consequents_readable,support,confidence,lift
0,{SWEETHEART CERAMIC TRINKET BOX},{STRAWBERRY CERAMIC TRINKET BOX},0.023613,0.721969,14.095205
1,{WOODEN FRAME ANTIQUE WHITE },{WOODEN PICTURE FRAME WHITE FINISH},0.028979,0.565445,11.772558
2,{WOODEN PICTURE FRAME WHITE FINISH},{WOODEN FRAME ANTIQUE WHITE },0.028979,0.603352,11.772558
3,{LOVE BUILDING BLOCK WORD},{HOME BUILDING BLOCK WORD},0.022778,0.528354,10.023487
4,{PACK OF 60 PINK PAISLEY CAKE CASES},{PACK OF 72 RETRO SPOT CAKE CASES},0.022092,0.528531,8.385737
5,{60 TEATIME FAIRY CAKE CASES},{PACK OF 72 RETRO SPOT CAKE CASES},0.024269,0.513565,8.148284
6,{LUNCH BAG WOODLAND},{LUNCH BAG RED SPOTTY},0.021943,0.552553,8.114346
7,{LUNCHBAG PINK RETROSPOT},{LUNCH BAG RED SPOTTY},0.027757,0.548292,8.051780
8,{LUNCH BAG BLACK SKULL.},{LUNCH BAG RED SPOTTY},0.028681,0.505783,7.427531
9,{JUMBO BAG STRAWBERRY},{JUMBO BAG RED WHITE SPOTTY },0.027250,0.632526,7.105008


In [53]:
gift_2_wrap = rules.sort_values('confidence', ascending=False).head(2)
gift_2_wrap[['antecedents_readable', 'consequents_readable', 'support', 'confidence', 'lift']]


,antecedents_readable,consequents_readable,support,confidence,lift
5,{SWEETHEART CERAMIC TRINKET BOX},{STRAWBERRY CERAMIC TRINKET BOX},0.023613,0.721969,14.095205
1,{RED HANGING HEART T-LIGHT HOLDER},{WHITE HANGING HEART T-LIGHT HOLDER},0.032676,0.707097,5.037539


### Part A Interpretation
The strongest rules by lift indicate tightly coupled product pairs. These signals are translated into concrete gift bundle proposals, balancing confidence and business usefulness.

In [54]:
last_date = df['InvoiceDate'].max()
df['Customer ID'] = df['Customer ID'].apply(int)
rfm = df.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (last_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('Price', lambda x: (x * df.loc[x.index, 'Quantity']).sum())
)
rfm.describe()


,Recency,Frequency,Monetary
count,5350.000000,5350.000000,5350.000000
mean,202.000935,6.269346,2751.990190
std,209.959231,11.995550,12080.466564
min,0.000000,1.000000,2.950000
25%,25.000000,1.000000,336.167500
50%,97.500000,3.000000,849.910000
75%,381.000000,7.000000,2214.905000
max,738.000000,336.000000,608821.650000


---
## Part B: Class Association Rules (CARs) - Customer Segmentation
Customers are labeled via RFM thresholds and then segment-specific CARs are mined to explain segment behavior and support first-order prediction.

In [75]:
R_q1, R_q3 = rfm['Recency'].quantile(0.25),   rfm['Recency'].quantile(0.75)
F_q1, F_q3 = rfm['Frequency'].quantile(0.25), rfm['Frequency'].quantile(0.75)
M_q1, M_q3 = rfm['Monetary'].quantile(0.25),  rfm['Monetary'].quantile(0.75)

print(f'Recency   Q1={R_q1}d, Q3={R_q3}d')
print(f'Frequency Q1={F_q1}, Q3={F_q3}')
print(f'Monetary  Q1={M_q1}, Q3={M_q3}')

def assign_segment(row):
    # Champion: recent (below Q1 recency = small gap), high freq & spend
    if row['Recency'] <= R_q1 and row['Frequency'] >= F_q3 and row['Monetary'] >= M_q3:
        return 'Champion'
    # One-and-Done: only 1 or 2 orders, very low monetary
    if row['Frequency'] <= 2 and row['Monetary'] <= M_q1:
        return 'One-and-Done'
    # At-Risk: not bought recently but used to buy regularly
    if row['Recency'] >= R_q3 and row['Frequency'] >= F_q1:
        return 'At-Risk'
    # Default remaining to At-Risk (middle ground)
    return 'At-Risk'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

print('\nSegment distribution:')
rfm['Segment'].value_counts()

Recency   Q1=25.0d, Q3=381.0d
Frequency Q1=1.0, Q3=7.0
Monetary  Q1=336.1675000000001, Q3=2214.905

Segment distribution:


Segment
At-Risk         3458
One-and-Done    1272
Champion         620
Name: count, dtype: int64

In [56]:
print('\n=== Per-Segment RFM Summary ===')
print(rfm.groupby('Segment')[['Recency','Frequency','Monetary']].mean().round(2))


=== Per-Segment RFM Summary ===
              Recency  Frequency  Monetary
Segment                                   
At-Risk        180.31       5.05   1955.04
Champion         9.35      23.48  12458.63
One-and-Done   354.88       1.18    187.30


In [57]:
df = df.merge(rfm['Segment'], left_on='Customer ID', right_index=True)
df.head()

,index,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Segment
0,0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,At-Risk
1,1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,At-Risk
2,2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,At-Risk
3,3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,At-Risk
4,4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,At-Risk


In [77]:
def mine_segment(segment_name):
    seg_df = df[df['Segment'] == segment_name]
    trans = seg_df.groupby('Invoice')['StockCode'].apply(list)

    te = TransactionEncoder()
    te_data = te.fit(trans).transform(trans)

    df_temp = pd.DataFrame(te_data, columns=te.columns_)

    freq = fpgrowth(df_temp, min_support=0.01, use_colnames=True)
    rules = association_rules(freq, metric="confidence", min_threshold=0.55)

    rules['segment'] = segment_name
    rules['antecedents_readable'] = rules['antecedents'].apply(decode_itemset)
    rules['consequents_readable'] = rules['consequents'].apply(decode_itemset)
    return rules

champion_rules = mine_segment('Champion')
atrisk_rules = mine_segment('At-Risk')
onedone_rules = mine_segment('One-and-Done')

In [59]:
print("\n=== Champion Rules ===")
display(champion_rules[['antecedents_readable', 'consequents_readable', 'support', 'confidence', 'lift']].sort_values('confidence', ascending=False).head(5).reset_index(drop=True))

print("\n=== At-Risk Rules ===")
display(atrisk_rules[['antecedents_readable', 'consequents_readable', 'support', 'confidence', 'lift']].sort_values('confidence', ascending=False).head(5).reset_index(drop=True))

print("\n=== One-and-Done Rules ===")
display(onedone_rules[['antecedents_readable', 'consequents_readable', 'support', 'confidence', 'lift']].sort_values('confidence', ascending=False).head(5).reset_index(drop=True))


=== Champion Rules ===


,antecedents_readable,consequents_readable,support,confidence,lift
0,{POPPY'S PLAYHOUSE LIVINGROOM },{POPPY'S PLAYHOUSE KITCHEN},0.010647,0.901163,57.039687
1,"{REGENCY CAKESTAND 3 TIER, PINK REGENCY TEACUP...",{GREEN REGENCY TEACUP AND SAUCER},0.010304,0.872093,32.305166
2,"{PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY...",{GREEN REGENCY TEACUP AND SAUCER},0.015181,0.870079,32.230550
3,"{GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY...",{ROSES REGENCY TEACUP AND SAUCER },0.015181,0.870079,28.528393
4,"{REGENCY CAKESTAND 3 TIER, PINK REGENCY TEACUP...",{ROSES REGENCY TEACUP AND SAUCER },0.010235,0.866279,28.403808



=== At-Risk Rules ===


,antecedents_readable,consequents_readable,support,confidence,lift
0,"{WOODEN PICTURE FRAME WHITE FINISH, WOOD 2 DRA...",{WOODEN FRAME ANTIQUE WHITE },0.011215,0.911628,17.469869
1,"{PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY...",{GREEN REGENCY TEACUP AND SAUCER},0.014076,0.911111,37.733386
2,{POPPY'S PLAYHOUSE LIVINGROOM },{POPPY'S PLAYHOUSE KITCHEN},0.010700,0.890476,55.383816
3,"{TOILET METAL SIGN, KITCHEN METAL SIGN}",{BATHROOM METAL SIGN},0.010356,0.887255,32.992668
4,"{KEY FOB , GARAGE DESIGN, KEY FOB , BACK DOOR }","{KEY FOB , SHED}",0.010128,0.876238,41.389203



=== One-and-Done Rules ===


,antecedents_readable,consequents_readable,support,confidence,lift
0,{WOODEN TREE CHRISTMAS SCANDINAVIAN},{WOODEN STAR CHRISTMAS SCANDINAVIAN},0.010624,0.888889,53.546667
1,{SET/10 BLUE SPOTTY PARTY CANDLES},{SET/10 PINK SPOTTY PARTY CANDLES},0.010624,0.800000,50.200000
2,{BAKING SET SPACEBOY DESIGN},{BAKING SET 9 PIECE RETROSPOT },0.018592,0.756757,13.898484
3,{EDWARDIAN PARASOL RED},{EDWARDIAN PARASOL BLACK},0.011952,0.750000,34.227273
4,{PLASTERS IN TIN WOODLAND ANIMALS},{PLASTERS IN TIN CIRCUS PARADE },0.010624,0.727273,32.213904


In [78]:
sample_items = {'82482', '22698', '22579'}

def predict_segment(rules_df):
    matched = []
    for _, row in rules_df.iterrows():
        if set(row['antecedents']).issubset(sample_items):
            matched.append((row['confidence'], row['segment']))
    return sorted(matched, reverse=True)

champion_probs = predict_segment(champion_rules)
atrisk_probs = predict_segment(atrisk_rules)
onedone_probs = predict_segment(onedone_rules)
print("Champion:", champion_probs)
print("At-Risk:", atrisk_probs)
print("One-and-Done:", onedone_probs)

Champion: [(0.8115015974440895, 'Champion'), (0.8115015974440895, 'Champion'), (0.7060702875399362, 'Champion')]
At-Risk: [(0.85, 'At-Risk'), (0.7946428571428571, 'At-Risk'), (0.7941176470588235, 'At-Risk'), (0.7235294117647058, 'At-Risk'), (0.6557971014492753, 'At-Risk'), (0.5647058823529412, 'At-Risk')]
One-and-Done: [(0.8888888888888888, 'One-and-Done')]


### Part B Interpretation
The rule-firing outputs above demonstrate how the first customer basket maps most strongly to At-Risk in this run. This supports using CAR confidence as a practical score for cold-start segment direction.

In [61]:
df = df.sort_values(['Customer ID', 'InvoiceDate'])

sequences = df.groupby(['Customer ID', 'Invoice'])['StockCode'].apply(list)
sequences = sequences.groupby('Customer ID').apply(list)

seq_list = sequences.tolist()

len(seq_list)

5350

---
## Part C: Sequential Pattern Mining - Customer Journey
This section converts invoice histories into ordered category events, mines frequent sequences, and derives at-risk trigger rules for retention action.

In [62]:
df.head()

,index,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Segment
19973,27994,491725,TEST001,This is a test product.,10,2009-12-14 08:34:00,4.5,12346,United Kingdom,At-Risk
20084,28251,491742,TEST001,This is a test product.,5,2009-12-14 11:00:00,4.5,12346,United Kingdom,At-Risk
20085,28254,491744,TEST001,This is a test product.,5,2009-12-14 11:02:00,4.5,12346,United Kingdom,At-Risk
26505,39398,492718,TEST001,This is a test product.,5,2009-12-18 10:47:00,4.5,12346,United Kingdom,At-Risk
26516,39411,492722,TEST002,This is a test product.,1,2009-12-18 10:55:00,1.0,12346,United Kingdom,At-Risk


In [ ]:
CATEGORY_RULES = [
    ('Candles & Home Fragrance',  ['CANDLE', 'FRAGRANCE', 'DIFFUSER', 'SCENT', 'WAX', 'AROMA', 'INCENSE']),
    ('Seasonal & Festive',        ['CHRISTMAS', 'XMAS', 'EASTER', 'HALLOWEEN', 'VALENTINES', 'FESTIVE',
                                   'SANTA', 'SNOWMAN', 'REINDEER', 'WREATH', 'BUNTING', 'GARLAND']),
    ('Kitchen & Dining',          ['MUG', 'CUP', 'PLATE', 'BOWL', 'GLASS', 'JUG', 'TEAPOT', 'CAKE',
                                   'BREAD', 'KITCHEN', 'DINING', 'SPOON', 'FORK', 'KNIFE', 'COFFEE']),
    ('Storage & Organisation',    ['BOX', 'BASKET', 'TIN', 'JAR', 'RACK', 'SHELF', 'STORAGE',
                                   'ORGANIS', 'HOLDER', 'TRAY', 'CHEST', 'CRATE']),
    ('Stationery & Gifts',        ['CARD', 'NOTEBOOK', 'PEN', 'PENCIL', 'GIFT', 'WRAP', 'BAG',
                                   'RIBBON', 'TAG', 'LABEL', 'STICKER', 'STAMP', 'LETTER']),
    ('Garden & Outdoor',          ['GARDEN', 'PLANT', 'POT', 'FLOWER', 'BIRD', 'OUTDOOR', 'WATERING',
                                   'SEED', 'TROWEL', 'PARASOL', 'FENCE']),
    ('Lighting & Decor',          ['LIGHT', 'LAMP', 'LANTERN', 'FAIRY', 'LED', 'MIRROR', 'FRAME',
                                   'CLOCK', 'DECOR', 'ORNAMENT', 'VASE', 'CUSHION', 'THROW']),
]

def map_category(desc):
    if pd.isna(desc):
        return 'Other'
    desc_upper = str(desc).upper()
    for cat, keywords in CATEGORY_RULES:
        if any(kw in desc_upper for kw in keywords):
            return cat
    return 'Other'

df['Category'] = df['Description'].apply(map_category)

cat_counts = df['Category'].value_counts()
covered    = df[df['Category'] != 'Other'].shape[0]
total      = df.shape[0]
print(f'Category coverage: {covered/total*100:.1f}% of transactions')
print(cat_counts.to_string())

Category coverage: 70.8% of transactions
Category
Other                       211829
Stationery & Gifts          127278
Kitchen & Dining            103703
Storage & Organisation       95206
Garden & Outdoor             61205
Seasonal & Festive           52586
Lighting & Decor             51887
Candles & Home Fragrance     21556


In [80]:
inv_cats = (
    df.groupby(['Customer ID', 'Invoice', df['InvoiceDate'].dt.date.rename('Date')])
      ['Category'].apply(lambda x: list(set(x)))
      .reset_index()
)
inv_cats.columns = ['Customer ID', 'Invoice', 'Date', 'Categories']
inv_cats = inv_cats.sort_values(['Customer ID', 'Date'])

# Build per-customer sequence (list of events = invoices; each event = list of categories)
cust_sequences = (
    inv_cats.groupby('Customer ID')['Categories']
    .apply(list)
    .reset_index()
)
cust_sequences.columns = ['Customer ID', 'Sequence']
print(f'Total customer sequences: {len(cust_sequences)}')
print(f'Avg sequence length (invoices): {cust_sequences["Sequence"].apply(len).mean():.2f}')

Total customer sequences: 5350
Avg sequence length (invoices): 6.27


In [81]:
from prefixspan import PrefixSpan
n_customers = len(cust_sequences)
abs_minsup  = max(1, int(0.005 * n_customers))
print(f'n_customers={n_customers}, abs_minsup={abs_minsup}')

# Flatten: encode categories as integers for PrefixSpan
all_cats = sorted(set(c for seq in cust_sequences['Sequence'] for event in seq for c in event))
cat_to_int = {c: i for i, c in enumerate(all_cats)}
int_to_cat = {i: c for c, i in cat_to_int.items()}

# PrefixSpan requires hashable sequence elements; represent each invoice-event as a tuple
ps_sequences = [
    [tuple(sorted(cat_to_int[c] for c in event)) for event in seq]
    for seq in cust_sequences['Sequence']
]

ps = PrefixSpan(ps_sequences)
freq_patterns = ps.frequent(abs_minsup)

print(f'Total frequent sequences found: {len(freq_patterns)}')

n_customers=5350, abs_minsup=26
Total frequent sequences found: 3933


In [ ]:
len1 = [(sup, pat) for sup, pat in freq_patterns if len(pat) == 1]
len2 = [(sup, pat) for sup, pat in freq_patterns if len(pat) == 2]
len3 = [(sup, pat) for sup, pat in freq_patterns if len(pat) == 3]

print(f'Length-1 sequences: {len(len1)}')
print(f'Length-2 sequences: {len(len2)}')
print(f'Length-3 sequences: {len(len3)}')

top10_len2 = sorted(len2, key=lambda x: -x[0])[:10]

print('\n=== Top 10 Frequent Length-2 Sequences ===')
print(f'{"#":<3} {"Sequence":<50} {"Support":>8} {"Support%":>8}  Interpretation')
print('-' * 110)
for i, (sup, pat) in enumerate(top10_len2, 1):
    decoded = [str([int_to_cat[x] for x in event]) for event in pat]
    seq_str = ' -> '.join(decoded)
    sup_pct = sup / n_customers * 100
    interp  = f'Customers who buy {int_to_cat[pat[0][0]]} later buy {int_to_cat[pat[1][0]]}'
    print(f'{i:<3} {seq_str:<50} {sup:>8} {sup_pct:>7.2f}%  {interp}')

Length-1 sequences: 141
Length-2 sequences: 925
Length-3 sequences: 1017

=== Top 10 Frequent Length-2 Sequences ===
#   Sequence                                            Support Support%  Interpretation
--------------------------------------------------------------------------------------------------------------
1   ['Candles & Home Fragrance', 'Garden & Outdoor', 'Kitchen & Dining', 'Lighting & Decor', 'Other', 'Seasonal & Festive', 'Stationery & Gifts', 'Storage & Organisation'] -> ['Candles & Home Fragrance', 'Garden & Outdoor', 'Kitchen & Dining', 'Lighting & Decor', 'Other', 'Seasonal & Festive', 'Stationery & Gifts', 'Storage & Organisation']      558   10.43%  Customers who buy Candles & Home Fragrance later buy Candles & Home Fragrance
2   ['Garden & Outdoor', 'Kitchen & Dining', 'Lighting & Decor', 'Other', 'Seasonal & Festive', 'Stationery & Gifts', 'Storage & Organisation'] -> ['Garden & Outdoor', 'Kitchen & Dining', 'Lighting & Decor', 'Other', 'Seasonal & Festive', 'Sta

In [67]:
at_risk_ids = set(df[df['Segment'] == 'At-Risk']['Customer ID'].astype(int))
print(f'At-Risk customers: {len(at_risk_ids)}')

at_risk_seqs = cust_sequences[cust_sequences['Customer ID'].isin(at_risk_ids)]
n_ar = len(at_risk_seqs)
print(f'At-Risk sequences: {n_ar}')

abs_minsup_ar = max(1, int(0.005 * n_ar))
print(f'abs_minsup for At-Risk: {abs_minsup_ar}')

ps_ar = [
    [tuple(sorted(cat_to_int[c] for c in event)) for event in seq]
    for seq in at_risk_seqs['Sequence']
]

ps2 = PrefixSpan(ps_ar)
freq_ar = ps2.frequent(abs_minsup_ar)
print(f'Frequent sequences in At-Risk subpopulation: {len(freq_ar)}')

len2_ar = sorted([(sup, pat) for sup, pat in freq_ar if len(pat) == 2], key=lambda x: -x[0])
len3_ar = sorted([(sup, pat) for sup, pat in freq_ar if len(pat) == 3], key=lambda x: -x[0])

print(f'Length-2 At-Risk sequences: {len(len2_ar)}')
print(f'Length-3 At-Risk sequences: {len(len3_ar)}')

print('\n=== Top 10 At-Risk Length-2 Sequences ===')
print(f'{"#":<3} {"Sequence":<55} {"Support":>8} {"Sup%":>6}')
print('-' * 95)
for i, (sup, pat) in enumerate(len2_ar[:10], 1):
    decoded = [str([int_to_cat[x] for x in event]) for event in pat]
    print(f'{i:<3} {" -> ".join(decoded):<55} {sup:>8} {100*sup/n_ar:>5.2f}%')

print('\n=== Top 5 At-Risk Length-3 Sequences ===')
print(f'{"#":<3} {"Sequence":<55} {"Support":>8} {"Sup%":>6}')
print('-' * 95)
for i, (sup, pat) in enumerate(len3_ar[:5], 1):
    decoded = [str([int_to_cat[x] for x in event]) for event in pat]
    print(f'{i:<3} {" -> ".join(decoded):<55} {sup:>8} {100*sup/n_ar:>5.2f}%')

At-Risk customers: 3458
At-Risk sequences: 3458
abs_minsup for At-Risk: 17
Frequent sequences in At-Risk subpopulation: 763
Length-2 At-Risk sequences: 391
Length-3 At-Risk sequences: 169

=== Top 10 At-Risk Length-2 Sequences ===
#   Sequence                                                 Support   Sup%
-----------------------------------------------------------------------------------------------
1   ['Candles & Home Fragrance', 'Garden & Outdoor', 'Kitchen & Dining', 'Lighting & Decor', 'Other', 'Seasonal & Festive', 'Stationery & Gifts', 'Storage & Organisation'] -> ['Candles & Home Fragrance', 'Garden & Outdoor', 'Kitchen & Dining', 'Lighting & Decor', 'Other', 'Seasonal & Festive', 'Stationery & Gifts', 'Storage & Organisation']      348 10.06%
2   ['Garden & Outdoor', 'Kitchen & Dining', 'Lighting & Decor', 'Other', 'Seasonal & Festive', 'Stationery & Gifts', 'Storage & Organisation'] -> ['Garden & Outdoor', 'Kitchen & Dining', 'Lighting & Decor', 'Other', 'Seasonal & Festive',

In [ ]:
champ_ids  = set(df[df['Segment']=='Champion']['Customer ID'].astype(int))
champ_seqs = cust_sequences[cust_sequences['Customer ID'].isin(champ_ids)]
n_ch       = len(champ_seqs)

print('Sequence length comparison (avg # of invoices):')
print(f'  At-Risk  : {at_risk_seqs["Sequence"].apply(len).mean():.2f} invoices/customer')
print(f'  Champions: {champ_seqs["Sequence"].apply(len).mean():.2f} invoices/customer')

# Build a lookup of At-Risk sequence support -> for C3 trigger rules
ar_sup_lookup = {str(pat): sup for sup, pat in freq_ar}

Sequence length comparison (avg # of invoices):
  At-Risk  : 5.05 invoices/customer
  Champions: 23.48 invoices/customer


In [69]:
def get_prefix_support(prefix_pat, sequences):
    """Count how many sequences start with prefix_pat."""
    count = 0
    for seq in sequences:
        if len(seq) >= len(prefix_pat):
            match = all(
                set(prefix_pat[i]).issubset(set(seq[i]))
                for i in range(len(prefix_pat))
            )
            if match:
                count += 1
    return count

# Pick top 3 length-2 at-risk sequences as trigger candidates
triggers = len2_ar[:3]

# Dataset date range for monthly estimate
total_months = (df['InvoiceDate'].max() - df['InvoiceDate'].min()).days / 30

print('=== Early-Warning Trigger Rules ===')
print()
for i, (full_sup, full_pat) in enumerate(triggers, 1):
    prefix     = [full_pat[0]]     # first event as trigger prefix
    prefix_sup = get_prefix_support(prefix, ps_ar)
    confidence = full_sup / prefix_sup if prefix_sup > 0 else 0

    prefix_cat  = [int_to_cat[x] for x in prefix[0]]
    next_cat    = [int_to_cat[x] for x in full_pat[1]]
    monthly_est = full_sup / total_months

    print(f'TRIGGER RULE {i}:')
    print(f'  IF  customer recently bought from: {prefix_cat}')
    print(f'  AND next purchase is from        : {next_cat}')
    print(f'  THEN flag as At-Risk for targeted discount voucher')
    print(f'  Sequence support  : {full_sup} customers ({full_sup/n_ar*100:.2f}% of At-Risk)')
    print(f'  Prefix support    : {prefix_sup} customers')
    print(f'  Confidence        : {confidence:.2%}')
    print(f'  Est. flagged/month: ~{monthly_est:.1f} customers')
    print(f'  Business Action   : Send a personalised 15% discount voucher within 7 days')
    print(f'                      of the trigger purchase event to encourage return visit.')
    print()

=== Early-Warning Trigger Rules ===

TRIGGER RULE 1:
  IF  customer recently bought from: ['Candles & Home Fragrance', 'Garden & Outdoor', 'Kitchen & Dining', 'Lighting & Decor', 'Other', 'Seasonal & Festive', 'Stationery & Gifts', 'Storage & Organisation']
  AND next purchase is from        : ['Candles & Home Fragrance', 'Garden & Outdoor', 'Kitchen & Dining', 'Lighting & Decor', 'Other', 'Seasonal & Festive', 'Stationery & Gifts', 'Storage & Organisation']
  THEN flag as At-Risk for targeted discount voucher
  Sequence support  : 348 customers (10.06% of At-Risk)
  Prefix support    : 453 customers
  Confidence        : 76.82%
  Est. flagged/month: ~14.1 customers
  Business Action   : Send a personalised 15% discount voucher within 7 days
                      of the trigger purchase event to encourage return visit.

TRIGGER RULE 2:
  IF  customer recently bought from: ['Garden & Outdoor', 'Kitchen & Dining', 'Lighting & Decor', 'Other', 'Seasonal & Festive', 'Stationery & Gifts', '

## Notebook Conclusion
The workflow forms one coherent pipeline: clean transactions -> FP-Growth bundles -> RFM segmentation + CARs -> sequence-based churn triggers. The same outputs are consolidated in the written report file for submission.